# Phase 3 — Customer Feature Engineering

**Goal:** Build a clean, reproducible customer-level feature table for unsupervised ML.

**Inputs**
- `data/processed/customer_sales.parquet`
- `data/processed/customer_returns.parquet`

**Outputs**
- `data/processed/customer_features.parquet` — raw/interpretable customer features
- `data/processed/customer_features_ml.parquet` — transformed + scaled features for clustering

> Run this notebook top-to-bottom after a kernel restart. It intentionally contains only the current feature-engineering logic; obsolete feature definitions have been removed.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## 2. Project Paths

In [2]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

SALES_PATH = PROCESSED_DATA_DIR / "customer_sales.parquet"
RETURNS_PATH = PROCESSED_DATA_DIR / "customer_returns.parquet"

CUSTOMER_FEATURES_PATH = PROCESSED_DATA_DIR / "customer_features.parquet"
CUSTOMER_FEATURES_ML_PATH = PROCESSED_DATA_DIR / "customer_features_ml.parquet"

print("Project root:", PROJECT_ROOT)
print("Sales file exists:", SALES_PATH.exists())
print("Returns file exists:", RETURNS_PATH.exists())

Project root: C:\Users\arman\OneDrive\Desktop\Data_Analyst\Projects\CustomerIQ_Unsupervised\CustomerIQ
Sales file exists: True
Returns file exists: True


## 3. Load Processed Data

In [3]:
sales_df = pd.read_parquet(SALES_PATH)
returns_df = pd.read_parquet(RETURNS_PATH)

print("Sales shape:", sales_df.shape)
print("Returns shape:", returns_df.shape)

Sales shape: (779425, 12)
Returns shape: (22496, 10)


## 4. Validate Inputs

In [4]:
required_sales_columns = {
    "Customer ID", "Invoice", "InvoiceDate",
    "Quantity", "Revenue", "StockCode"
}

required_return_columns = {
    "Customer ID", "Invoice", "Quantity", "ReturnValue"
}

missing_sales = required_sales_columns - set(sales_df.columns)
missing_returns = required_return_columns - set(returns_df.columns)

assert not missing_sales, f"Missing sales columns: {missing_sales}"
assert not missing_returns, f"Missing return columns: {missing_returns}"

assert sales_df["Customer ID"].isna().sum() == 0
assert (sales_df["Quantity"] <= 0).sum() == 0
assert (sales_df["Revenue"] <= 0).sum() == 0

print("Input validation passed.")

Input validation passed.


## 5. Reference Date for Recency

Use the day after the latest valid purchase in the dataset rather than today's date. This makes historical RFM recency meaningful.

In [5]:
reference_date = (
    sales_df["InvoiceDate"].max()
    + pd.Timedelta(days=1)
)

print("Latest purchase:", sales_df["InvoiceDate"].max())
print("Reference date:", reference_date)

Latest purchase: 2011-12-09 12:50:00
Reference date: 2011-12-10 12:50:00


## 6. Build Base Customer Features

One row represents one customer.

We retain `total_quantity` because it is needed to calculate `avg_items_per_order`, but it is not part of the final clustering feature set.

In [6]:
customer_features = (
    sales_df
    .groupby("Customer ID")
    .agg(
        first_purchase=("InvoiceDate", "min"),
        last_purchase=("InvoiceDate", "max"),
        frequency=("Invoice", "nunique"),
        monetary=("Revenue", "sum"),
        total_quantity=("Quantity", "sum"),
        unique_products=("StockCode", "nunique")
    )
    .reset_index()
)

customer_features["recency_days"] = (
    reference_date - customer_features["last_purchase"]
).dt.total_seconds() / 86400

customer_features["purchase_span_days"] = (
    customer_features["last_purchase"] -
    customer_features["first_purchase"]
).dt.total_seconds() / 86400

customer_features["average_order_value"] = (
    customer_features["monetary"] /
    customer_features["frequency"]
)

customer_features["avg_items_per_order"] = (
    customer_features["total_quantity"] /
    customer_features["frequency"]
)

print("Base customer feature shape:", customer_features.shape)

Base customer feature shape: (5878, 11)


## 7. Build Customer Return Features

Only return invoices and return value are retained for the final return-behavior features. Raw return counts/quantity are not used in clustering.

In [7]:
return_features = (
    returns_df
    .dropna(subset=["Customer ID"])
    .groupby("Customer ID")
    .agg(
        return_invoices=("Invoice", "nunique"),
        return_value=("ReturnValue", "sum")
    )
    .reset_index()
)

print("Return feature shape:", return_features.shape)
return_features.head()

Return feature shape: (2572, 3)


,Customer ID,return_invoices,return_value
0,12346,5,"77,608.20"
1,12349,1,24.15
2,12352,3,960.63
3,12359,4,221.05
4,12360,1,40.00


## 8. Merge Return Features

In [8]:
customer_features = customer_features.merge(
    return_features,
    on="Customer ID",
    how="left"
)

customer_features[["return_invoices", "return_value"]] = (
    customer_features[["return_invoices", "return_value"]].fillna(0)
)

print("After return merge:", customer_features.shape)

After return merge: (5878, 13)


## 9. Create Return Behavior Features

The shares are bounded between 0 and 1 and are easier to interpret than the previous `return_rate` / `return_value_rate` definitions.

In [9]:
customer_features["return_order_share"] = (
    customer_features["return_invoices"] /
    (
        customer_features["return_invoices"] +
        customer_features["frequency"]
    )
)

customer_features["return_value_share"] = (
    customer_features["return_value"] /
    (
        customer_features["return_value"] +
        customer_features["monetary"]
    )
)

customer_features["has_return"] = (
    customer_features["return_invoices"] > 0
).astype(int)

print(
    customer_features[
        [
            "return_invoices",
            "return_value",
            "return_order_share",
            "return_value_share",
            "has_return"
        ]
    ].describe().T
)

                      count   mean      std  min  25%  50%   75%        max
return_invoices    5,878.00   1.33     3.67 0.00 0.00 0.00  1.00     112.00
return_value       5,878.00 174.17 2,685.07 0.00 0.00 0.00 30.79 168,478.60
return_order_share 5,878.00   0.12     0.17 0.00 0.00 0.00  0.22       0.80
return_value_share 5,878.00   0.02     0.07 0.00 0.00 0.00  0.02       0.75
has_return         5,878.00   0.43     0.49 0.00 0.00 0.00  1.00       1.00


## 10. Validate Customer Feature Table

In [10]:
print("Shape:", customer_features.shape)

print("\nMissing values:")
print(customer_features.isna().sum())

print("\nCustomers:", customer_features["Customer ID"].nunique())

assert customer_features["Customer ID"].is_unique
assert customer_features.isna().sum().sum() == 0

print("\nFeature validation passed.")

Shape: (5878, 16)

Missing values:
Customer ID            0
first_purchase         0
last_purchase          0
frequency              0
monetary               0
total_quantity         0
unique_products        0
recency_days           0
purchase_span_days     0
average_order_value    0
avg_items_per_order    0
return_invoices        0
return_value           0
return_order_share     0
return_value_share     0
has_return             0
dtype: int64

Customers: 5878

Feature validation passed.


## 11. Select Features for Clustering

The baseline clustering representation uses 10 behavioral features. Redundant features such as `active_days`, `avg_days_between_orders`, `product_diversity_ratio`, raw return counts/quantity, and the old return-rate variables are intentionally excluded.

In [11]:
selected_features = [
    "recency_days",
    "frequency",
    "monetary",
    "average_order_value",
    "avg_items_per_order",
    "unique_products",
    "purchase_span_days",
    "return_order_share",
    "return_value_share",
    "has_return"
]

model_features = customer_features[
    ["Customer ID"] + selected_features
].copy()

print("ML feature table shape:", model_features.shape)
model_features.head()

ML feature table shape: (5878, 11)


,Customer ID,recency_days,frequency,monetary,average_order_value,avg_items_per_order,unique_products,purchase_span_days,return_order_share,return_value_share,has_return
0,12346,326.12,12,"77,556.46","6,463.04","6,190.42",27,400.06,0.29,0.50,1
1,12347,2.87,8,"4,921.53",615.19,370.88,126,402.06,0.00,0.00,0
2,12348,75.98,5,"2,019.40",403.88,542.80,25,362.93,0.00,0.00,0
3,12349,19.12,4,"4,428.69","1,107.17",406.00,138,570.85,0.20,0.01,1
4,12350,310.87,1,334.40,334.40,197.00,17,0.00,0.00,0.00,0


## 12. Check Feature Skew

In [12]:
model_skewness = (
    model_features[selected_features]
    .skew()
    .sort_values(ascending=False)
)

model_skewness.to_frame("skewness")

,skewness
average_order_value,57.13
avg_items_per_order,47.91
monetary,25.07
frequency,12.64
unique_products,6.22
return_value_share,5.51
return_order_share,1.26
recency_days,0.89
purchase_span_days,0.39
has_return,0.29


## 13. Log-Transform Strongly Skewed Positive Features

`log1p` is used only for the features where the earlier EDA showed severe right skew. Recency, purchase span, return shares, and the binary return flag remain on their original scales until standardization.

In [13]:
log_features = [
    "frequency",
    "monetary",
    "average_order_value",
    "avg_items_per_order",
    "unique_products"
]

model_features_transformed = model_features.copy()

for feature in log_features:
    model_features_transformed[f"log_{feature}"] = np.log1p(
        model_features_transformed[feature]
    )

transformed_skewness = (
    model_features_transformed[
        [f"log_{feature}" for feature in log_features]
    ]
    .skew()
)

comparison = pd.DataFrame({
    "original_skew": model_features[log_features].skew(),
    "transformed_skew": transformed_skewness
})

comparison

,original_skew,transformed_skew
average_order_value,57.13,NaN
avg_items_per_order,47.91,NaN
frequency,12.64,NaN
log_average_order_value,NaN,0.06
log_avg_items_per_order,NaN,-0.63
log_frequency,NaN,1.00
log_monetary,NaN,0.27
log_unique_products,NaN,-0.28
monetary,25.07,NaN
unique_products,6.22,NaN


## 14. Build the Final Modeling Matrix

For each log-transformed feature, use the transformed version instead of the raw version. Keep the other five selected features unchanged. `Customer ID` is retained separately for joining predictions/results and is never used as a clustering feature.

In [14]:
final_model_columns = [
    "recency_days",
    "log_frequency",
    "log_monetary",
    "log_average_order_value",
    "log_avg_items_per_order",
    "log_unique_products",
    "purchase_span_days",
    "return_order_share",
    "return_value_share",
    "has_return"
]

X = model_features_transformed[final_model_columns].copy()

print("Final unscaled matrix:", X.shape)
print("Missing values:", X.isna().sum().sum())

Final unscaled matrix: (5878, 10)
Missing values: 0


## 15. Standardize Features

After the skewed features are compressed with `log1p`, standardization puts all behavioral dimensions onto a comparable scale for distance-based clustering algorithms such as K-Means.

In [15]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=final_model_columns,
    index=model_features_transformed["Customer ID"]
)

print("Scaled matrix shape:", X_scaled_df.shape)
print("\nMean (approximately 0):")
print(X_scaled_df.mean().round(4))

print("\nStd (approximately 1):")
print(X_scaled_df.std().round(4))

Scaled matrix shape: (5878, 10)

Mean (approximately 0):
recency_days               0.00
log_frequency              0.00
log_monetary               0.00
log_average_order_value    0.00
log_avg_items_per_order   -0.00
log_unique_products       -0.00
purchase_span_days         0.00
return_order_share        -0.00
return_value_share         0.00
has_return                -0.00
dtype: float64

Std (approximately 1):
recency_days              1.00
log_frequency             1.00
log_monetary              1.00
log_average_order_value   1.00
log_avg_items_per_order   1.00
log_unique_products       1.00
purchase_span_days        1.00
return_order_share        1.00
return_value_share        1.00
has_return                1.00
dtype: float64


## 16. Save Phase 3 Outputs

Two files are saved:

- `customer_features.parquet`: interpretable/raw customer-level features.
- `customer_features_ml.parquet`: transformed + standardized features for unsupervised ML.

In [16]:
customer_features.to_parquet(
    CUSTOMER_FEATURES_PATH,
    index=False
)

customer_features_ml = X_scaled_df.reset_index()

customer_features_ml.to_parquet(
    CUSTOMER_FEATURES_ML_PATH,
    index=False
)

print("Saved:")
print(CUSTOMER_FEATURES_PATH)
print(CUSTOMER_FEATURES_ML_PATH)

Saved:
C:\Users\arman\OneDrive\Desktop\Data_Analyst\Projects\CustomerIQ_Unsupervised\CustomerIQ\data\processed\customer_features.parquet
C:\Users\arman\OneDrive\Desktop\Data_Analyst\Projects\CustomerIQ_Unsupervised\CustomerIQ\data\processed\customer_features_ml.parquet


## 17. Final Validation

In [17]:
saved_raw = pd.read_parquet(CUSTOMER_FEATURES_PATH)
saved_ml = pd.read_parquet(CUSTOMER_FEATURES_ML_PATH)

print("Raw customer features:", saved_raw.shape)
print("ML-ready features:", saved_ml.shape)

print("\nML columns:")
print(saved_ml.columns.tolist())

assert saved_raw["Customer ID"].is_unique
assert saved_ml["Customer ID"].is_unique
assert saved_ml.isna().sum().sum() == 0

print("\nPhase 3 feature engineering completed successfully.")

Raw customer features: (5878, 16)
ML-ready features: (5878, 11)

ML columns:
['Customer ID', 'recency_days', 'log_frequency', 'log_monetary', 'log_average_order_value', 'log_avg_items_per_order', 'log_unique_products', 'purchase_span_days', 'return_order_share', 'return_value_share', 'has_return']

Phase 3 feature engineering completed successfully.
